# Lesson 3.2 — Writing Tool Descriptions and Building Tool Sets

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Write tool descriptions that improve selection accuracy
- Design input schemas with helpful field descriptions and enums
- Curate a tool set that avoids overlap and ambiguity
- Handle tool errors with clear, actionable messages

**You will produce:** A curated multi-tool set with strong descriptions and an error-handling wrapper.

## Concepts

A tool's **description** is prompt engineering for the model's tool selection. Good descriptions:

- State **what** the tool does, **when** to use it, and **when not** to.
- Describe each input field (type, format, units, examples) and use `enum` for fixed choices.
- Avoid overlap: two tools with similar descriptions cause mis-selection. Make boundaries crisp.
- **Error handling**: return structured, human-readable errors (not stack traces) so Claude can recover or ask for clarification.

A **tool set** should be minimal and orthogonal - more tools means more tokens and more chances to pick the wrong one.

> **Reference:** [Anthropic - Tool use best practices](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/implement-tool-use)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Weak vs strong descriptions

Compare an ambiguous description with a precise one.

In [ ]:
weak = {"name": "search", "description": "search stuff",
        "input_schema": {"type": "object",
            "properties": {"q": {"type": "string"}}, "required": ["q"]}}

strong = {"name": "search_orders",
    "description": ("Search the ORDER database by customer email or order id. "
        "Use ONLY for order lookups, not product catalog search."),
    "input_schema": {"type": "object", "properties": {
        "query": {"type": "string", "description": "customer email or order id"},
        "status": {"type": "string", "enum": ["open", "shipped", "all"],
            "description": "filter by order status"}},
        "required": ["query"]}}
print("strong tool:", strong["name"])

## Code-Along: Step 2 — Error-handling wrapper

Wrap execution so every tool returns a consistent result or a clear error.

In [ ]:
def safe_tool(fn):
    def wrapper(**kwargs):
        try:
            return {"ok": True, "result": fn(**kwargs)}
        except Exception as e:
            return {"ok": False, "error": f"{type(e).__name__}: {e}"}
    return wrapper

@safe_tool
def divide(a, b):
    return a / b

print(divide(a=10, b=2))
print(divide(a=10, b=0))  # returns a clean error, not a crash

## Exercise

Design a 3-tool set for a travel assistant (`search_flights`, `book_flight`, `cancel_booking`) with non-overlapping descriptions and enums where appropriate. Write the dicts; no API call needed.

In [ ]:
TOOLS = [
    # TODO: search_flights (origin, destination, date, cabin enum)
    # TODO: book_flight (flight_id, passenger)
    # TODO: cancel_booking (booking_id)
]
# TODO: assert each description says when to use and when NOT to use it

## Key Takeaways

- Tool descriptions guide selection: say what it does and when (not) to use it.
- Describe every field; use enums for fixed choices.
- Keep tool sets minimal and orthogonal to avoid mis-selection.
- Return structured, readable errors so Claude can recover.

---

**Next:** [Lesson 3.3 — Client-Side vs Server-Side Tools and Approval Patterns](./lesson-3.3-client-server-tools-approval.ipynb)